In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

In [2]:
# objects.json is always next to this notebook
data_path = Path("objects.json")
with data_path.open("r", encoding="utf-8") as f:
  raw = json.load(f)

In [3]:
df = pd.DataFrame.from_dict(raw, orient="index").reset_index(drop=True)

In [4]:
df.shape

(4200, 54)

In [5]:
df.columns

Index(['objectID', 'isHighlight', 'accessionNumber', 'accessionYear',
       'isPublicDomain', 'primaryImage', 'primaryImageSmall',
       'additionalImages', 'constituents', 'department', 'objectName', 'title',
       'culture', 'period', 'objectDate', 'objectBeginDate', 'objectEndDate',
       'medium', 'dimensions', 'measurements', 'creditLine', 'classification',
       'metadataDate', 'repository', 'objectURL', 'tags', 'isTimelineWork',
       'objectWikidata_URL', 'GalleryNumber', 'artistRole',
       'artistDisplayName', 'artistDisplayBio', 'artistAlphaSort',
       'artistNationality', 'artistBeginDate', 'artistEndDate',
       'artistWikidata_URL', 'artistULAN_URL', 'artistPrefix', 'country',
       'geographyType', 'state', 'region', 'city', 'river', 'artistSuffix',
       'dynasty', 'subregion', 'locale', 'locus', 'excavation', 'reign',
       'county', 'artistGender'],
      dtype='object')

In [6]:
df.head()


,objectID,isHighlight,accessionNumber,accessionYear,isPublicDomain,primaryImage,primaryImageSmall,additionalImages,constituents,department,...,river,artistSuffix,dynasty,subregion,locale,locus,excavation,reign,county,artistGender
0,44793,False,50.61.11,1950,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,44817,False,1975.268.185,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[https://images.metmuseum.org/CRDImages/as/ori...,None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,44830,False,1975.268.378,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,44862,False,1975.268.473,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[https://images.metmuseum.org/CRDImages/as/ori...,None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,44907,False,1975.268.376,1975,True,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...,[],None,Asian Art,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# If your column is named differently, replace "classification" below.
col = "classification"

# Normalize blanks to NA first
s = (
    df[col]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Overall coverage
pct_has_classification = s.notna().mean() * 100
print(f"Rows with classification: {s.notna().sum()} / {len(df)} ({pct_has_classification:.2f}%)")

# Per-classification counts + percent of all rows
classification_summary = (
    s.fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis(col)
    .reset_index(name="count")
)

classification_summary["percent_of_all_rows"] = (
    classification_summary["count"] / len(df) * 100
).round(2)

classification_summary

Rows with classification: 4098 / 4200 (97.57%)


,classification,count,percent_of_all_rows
0,Ceramics,2548,60.67
1,Ceramics-Containers,524,12.48
2,Ceramics-Pottery,427,10.17
3,Ceramics-Porcelain,297,7.07
4,Vases,257,6.12
5,MISSING,102,2.43
6,Ceramics-Porcelain-Export,32,0.76
7,Ceramics-Vessels,8,0.19
8,Ceramics-Faience,3,0.07
9,Terracottas,2,0.05


In [8]:
missing_classification_rows = (
    df.loc[
        df["classification"].astype("string").str.strip().isna()
        | (df["classification"].astype("string").str.strip() == ""),
        ["department", "objectID", "title"],
    ]
    .sort_values(["department", "objectID"])
    .reset_index(drop=True)
)

# show every row/column in notebook output
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(missing_classification_rows)

,department,objectID,title
0,Ancient West Asian Art,891616,Jug with Assyrian-inspired decoration
1,Ancient West Asian Art,891620,Jug with Assyrian-inspired decoration
2,Egyptian Art,545042,Tell el-Yahudiya type juglet
3,Egyptian Art,545772,Classic Kerma Beaker
4,Egyptian Art,545791,Two-handed pottery vase of Amenhotep
5,Egyptian Art,546034,Black-topped red ware jar
6,Egyptian Art,546649,Jar
7,Egyptian Art,546758,White cross-lined ware beaker with hippos
8,Egyptian Art,546986,Cypriot ring-based Juglet
9,Egyptian Art,547254,Jar Decorated with Boats


In [9]:
target_classification = "Ceramics-Porcelain-Export"

rows = (
    df.loc[
        df["classification"].astype("string").str.strip() == target_classification,
        ["department", "objectID", "title"],
    ]
    .sort_values(["department", "objectID"])
    .reset_index(drop=True)
)

# show all rows
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(rows)

,department,objectID,title
0,European Sculpture and Decorative Arts,185900,Puzzle jug
1,European Sculpture and Decorative Arts,185901,Vase
2,European Sculpture and Decorative Arts,185902,Jug with cover
3,European Sculpture and Decorative Arts,185914,Jug with cover
4,European Sculpture and Decorative Arts,194624,Pitcher with cover
5,European Sculpture and Decorative Arts,195127,Jug
6,European Sculpture and Decorative Arts,195307,Jug
7,European Sculpture and Decorative Arts,200670,Vase
8,European Sculpture and Decorative Arts,201054,Milk jug with cover
9,European Sculpture and Decorative Arts,201289,Cream jug


### Map
Possible columns:
- department
- culture
- country
- state
- region
- city
- geographyType

In [10]:
cols_to_check = [
    "department",
    "culture",
    "country",
    "state",
    "region",
    "city",
    "geographyType",
    "artistNationality",
]

results = []

for col in cols_to_check:
    s = df[col]

    # Start with nulls
    missing = s.isna()

    # For string-like values, also treat blank/whitespace/"undefined" as missing
    s_str = s.astype("string").str.strip()
    missing = missing | s_str.eq("") | s_str.str.lower().eq("undefined")

    present = ~missing

    total = len(s)
    present_count = int(present.sum())
    missing_count = int(missing.sum())

    results.append({
        "column": col,
        # "present_count": present_count,
        # "missing_count": missing_count,
        "present_pct": round(present_count / total * 100, 2),
        # "missing_pct": round(missing_count / total * 100, 2),
    })

summary = pd.DataFrame(results).sort_values("present_pct", ascending=False)
summary

,column,present_pct
0,department,100.00
1,culture,73.79
2,country,26.71
6,geographyType,14.17
7,artistNationality,10.48
5,city,7.98
4,region,5.19
3,state,4.88


In [11]:
# Count of rows per department
department_counts = (
    df["department"]
    .fillna("<<MISSING_DEPARTMENT>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_DEPARTMENT>>")
    .value_counts(dropna=False)
)

# department_counts

department_summary = pd.DataFrame({
    "count": department_counts,
    "percent": (department_counts / department_counts.sum() * 100).round(2)
})

department_summary

,count,percent
department,,
Asian Art,1998,47.57
European Sculpture and Decorative Arts,690,16.43
The Michael C. Rockefeller Wing,531,12.64
Islamic Art,371,8.83
Greek and Roman Art,259,6.17
Medieval Art,128,3.05
Robert Lehman Collection,76,1.81
The American Wing,68,1.62
The Cloisters,34,0.81


In [12]:
# For each department, compute % present for every column in cols_to_check

def is_present(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    return series.notna() & s.ne("") & s.str.lower().ne("undefined")

# Clean department labels for grouping
dept_series = (
    df["department"]
    .fillna("<<MISSING_DEPARTMENT>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_DEPARTMENT>>")
)

# Build a table: rows = department, cols = indicators for presence
out = pd.DataFrame({"department": dept_series})

for col in cols_to_check:
    out[f"{col}_present"] = is_present(df[col]).astype(int)

dept_present_pct = (
    out.groupby("department")[[f"{c}_present" for c in cols_to_check]]
    .mean()
    .mul(100)
    .round(0)
)

# rename "country_present" -> "country", etc.
dept_present_pct.columns = [c.replace("_present", "") for c in dept_present_pct.columns]

# add total rows per department
dept_counts = out.groupby("department").size().rename("total_count")
dept_present_pct = dept_present_pct.join(dept_counts)

# optional: show count first
dept_present_pct = dept_present_pct[["total_count"] + cols_to_check]

dept_present_pct

,total_count,department,culture,country,state,region,city,geographyType,artistNationality
department,,,,,,,,,
Ancient West Asian Art,2,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
Asian Art,1998,100.0,100.0,0.0,0.0,0.0,0.0,0.0,2.0
Egyptian Art,32,100.0,0.0,97.0,0.0,53.0,0.0,100.0,0.0
European Sculpture and Decorative Arts,690,100.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0
Greek and Roman Art,259,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0
Islamic Art,371,100.0,0.0,100.0,0.0,0.0,62.0,100.0,0.0
Medieval Art,128,100.0,100.0,73.0,63.0,0.0,21.0,74.0,0.0
Modern and Contemporary Art,11,100.0,45.0,0.0,0.0,0.0,0.0,0.0,100.0
Robert Lehman Collection,76,100.0,99.0,0.0,0.0,0.0,0.0,0.0,1.0


Findings:
- 100% have "department"
- 71.10% have "culture"
- 25.12% have "country"


Location by department:
- Asian Art: culture
- European Sculpture and Decorative Arts: artistNationality
- Islamic Art: city, country
- Medieval Art: city, state, country, culture
- Modern and Contemporary Art: culture, artistNationality
- Robert Lehman Collection: culture, artistNationality
- The Cloisters: city, state, country, culture
- The Michael C. Rockefeller Wing: city, state, country, culture

In [13]:
# Make a copy
df_loc = df.copy()

def clean_missing(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    return s.mask(s.eq("") | s.str.lower().eq("undefined"))

# Department -> ordered source columns for location
location_priority = {
    "Asian Art": ["culture"],
    "European Sculpture and Decorative Arts": ["artistNationality"],
    "Islamic Art": ["city", "country"],
    "Medieval Art": ["city", "state", "country", "culture"],
    "Modern and Contemporary Art": ["culture", "artistNationality"],
    "Robert Lehman Collection": ["culture", "artistNationality"],
    "The Cloisters": ["city", "state", "country", "culture"],
    "The Michael C. Rockefeller Wing": ["city", "state", "country", "culture"],
    "Egyptian Art": ["country", "region"],
    "Greek and Roman Art": ["culture"]
}

# Start empty
df_loc["location"] = pd.NA

# Fill location by department using first non-missing column in priority order
for dept, cols in location_priority.items():
    mask = df_loc["department"].eq(dept)
    if not mask.any():
        continue

    # build first-non-missing across preferred columns
    candidates = pd.DataFrame({c: clean_missing(df_loc.loc[mask, c]) for c in cols})
    df_loc.loc[mask, "location"] = candidates.bfill(axis=1).iloc[:, 0]

# quick check
# df_loc[["department", "culture", "artistNationality", "city", "state", "country", "location"]].head(20)

In [14]:
# How many got location per listed department
check = (
    df_loc[df_loc["department"].isin(location_priority)]
    .assign(location_present=lambda d: d["location"].notna())
    .groupby("department")["location_present"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)
check

department
Asian Art                                 100.00
Egyptian Art                              100.00
European Sculpture and Decorative Arts    100.00
Greek and Roman Art                       100.00
Medieval Art                              100.00
Modern and Contemporary Art               100.00
Robert Lehman Collection                  100.00
The Cloisters                             100.00
The Michael C. Rockefeller Wing           100.00
Islamic Art                                99.73
Name: location_present, dtype: float64

In [15]:
location_counts = (
    df_loc["location"]
    .fillna("<<MISSING_LOCATION>>")
    .astype("string")
    .str.strip()
    .replace("", "<<MISSING_LOCATION>>")
    .value_counts(dropna=False)
)

location_summary = pd.DataFrame({
    # "count": location_counts,
    "percent": (location_counts / location_counts.sum() * 100).round(2)
})

location_summary

,percent
location,
China,30.21
Japan,13.86
Peru,10.21
British,7.57
French,4.05
...,...
French (Limoges),0.02
Wiltshire,0.02
Tembladera,0.02


In [16]:
n_unique_locations = (
    df_loc["location"]
    .fillna("")
    .astype("string")
    .str.strip()
    .replace("", "")
    .nunique(dropna=False)
)

print(f"Number of different locations: {n_unique_locations:,}")

Number of different locations: 199


In [17]:
all_locations = (
    df_loc["location"]
    .fillna("")
    .astype("string")
    .str.strip()
    .replace("", "")
    .drop_duplicates()
    .sort_values()
    .tolist()
)

all_locations

['',
 'Abu Mena',
 'Alta Verapaz',
 'American',
 'American and French',
 'Ancash',
 'Arizona',
 'Bandiagara Escarpment',
 'Belgian',
 'Bohemian',
 'British',
 'British, Scottish',
 'Byzantine (Egypt)',
 'Canosan, Puglia',
 'Chiclayo',
 'Chihuahua',
 'China',
 'China (?)',
 'China or Japan (?)',
 'ChinaNeihuLu',
 'Chinese',
 'Cocle Province',
 'Colima',
 'Colombia',
 'Comala',
 'Ctesiphon',
 'Cycladic',
 'Cycladic or Cretan',
 'Cypriot',
 'Danish',
 'Danish (Nästved)',
 'Derbyshire',
 'Dutch',
 'Dutch (Delft)',
 'East Greek',
 'East Greek, Rhodian',
 'Eastern Syria',
 'Ecuador',
 'Egypt',
 'Egyptian',
 'Etruscan',
 'Etruscan, Etrusco-Corinthian',
 'Etruscan, Italo-Corinthian',
 'Europe',
 'European',
 'Faliscan',
 'Florence',
 'Florence or its vicinity',
 'French',
 'French (Limoges)',
 'French (Paris)',
 'Fustat',
 'German',
 'Greek',
 'Greek or Roman',
 'Greek, Asia Minor',
 'Greek, Attic',
 'Greek, Boeotian (or Attic)',
 'Greek, Chalcidian',
 'Greek, Corinthian',
 'Greek, Egypt, Alex

### Timeline
- objectBeginDate
- objectEndDate
- objectDate

In [18]:
date_cols = ["objectBeginDate", "objectEndDate", "objectDate"]

rows = []
for col in date_cols:
    s = df[col]
    s_str = s.astype("string").str.strip()
    present = s.notna() & s_str.ne("") & s_str.str.lower().ne("undefined")
    rows.append({
        "column": col,
        "present_count": int(present.sum()),
        "missing_count": int((~present).sum()),
        "present_pct": round(present.mean() * 100, 2),
    })

pd.DataFrame(rows)

,column,present_count,missing_count,present_pct
0,objectBeginDate,4200,0,100.00
1,objectEndDate,4200,0,100.00
2,objectDate,3830,370,91.19


In [19]:
b = pd.to_numeric(df["objectBeginDate"], errors="coerce")
e = pd.to_numeric(df["objectEndDate"], errors="coerce")

both_present = b.notna() & e.notna()
equal = both_present & (b == e)

n_total = len(df)
n_both = int(both_present.sum())
n_equal = int(equal.sum())

print(f"Rows where begin & end are both present:  ({n_both / n_total * 100:.2f}%)")
print(f"Rows where objectBeginDate == objectEndDate (and both present): ({n_equal / n_total * 100:.2f}%)")
# print(f"Among rows with both present: {n_equal / n_both * 100:.2f}%" if n_both else "Among rows with both present: n/a (no rows)")

Rows where begin & end are both present:  (100.00%)
Rows where objectBeginDate == objectEndDate (and both present): (6.40%)


In [20]:
df[date_cols].sample(n=min(20, len(df)), random_state=42)

,objectBeginDate,objectEndDate,objectDate
1743,-500,-200,5th–3rd century BCE
2196,1695,1710,ca. 1700–1705
1728,1300,1399,14th century
3337,1622,1722,NaN
298,1600,1699,17th century
1837,1700,1799,18th century
4016,1640,1660,ca. 1650
351,1100,1299,12th–13th century
3315,1400,1599,15th–16th century
2458,1890,1900,ca. 1895


In [21]:
df2 = df.copy()
b = pd.to_numeric(df2["objectBeginDate"], errors="coerce")
e = pd.to_numeric(df2["objectEndDate"], errors="coerce")
mid = ((b + e) / 2).where(b.notna() & e.notna())
df2["final_date"] = (mid // 1).where(mid.notna()).astype("Int64")

In [22]:
df2['final_date'].sample(n=min(20, len(df)), random_state=42)

1743    -350
2196    1702
1728    1349
3337    1672
298     1649
1837    1749
4016    1650
351     1199
3315    1499
2458    1895
1029    1583
2319    1849
1477    1807
3773    -413
2902    -600
2292    1749
3571    1600
497     1319
1995    1700
734     1780
Name: final_date, dtype: Int64

In [23]:
df2["final_date"].agg(["min", "max"])

min   -4250
max    1950
Name: final_date, dtype: int64

# Get Function

In [24]:
# Normalize objectName first
obj_name = (
    df["objectName"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

objectname_summary = (
    obj_name.fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis("objectName")
    .reset_index(name="count")
)

objectname_summary["percent"] = (
    objectname_summary["count"] / len(df) * 100
).round(2)

# Show all distinct values with count + percent
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(objectname_summary)

,objectName,count,percent
0,Vase,1136,27.05
1,Bottle,720,17.14
2,Jar,564,13.43
3,Jug,246,5.86
4,Ewer,210,5.0
5,Tea jar,94,2.24
6,Vessel,80,1.9
7,Covered jar,78,1.86
8,Pitcher,59,1.4
9,Bottle vase,47,1.12


In [25]:
df["objectName"].astype("string").str.strip().replace("", pd.NA).notna().mean() * 100

np.float64(100.0)

In [30]:
# Normalize objectName first
title = (
    df["title"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

title_summary = (
    title.fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis("title")
    .reset_index(name="count")
)

title_summary["percent"] = (
    title_summary["count"] / len(df) * 100
).round(2)

# Show all distinct values with count + percent
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(title_summary)

,title,count,percent
0,Vase,574,13.67
1,Jar,294,7.0
2,Bottle,226,5.38
3,Jug,184,4.38
4,Ewer,145,3.45
5,Tea Jar,58,1.38
6,Pitcher,55,1.31
7,Vase with cover,35,0.83
8,Covered Jar,35,0.83
9,Tea jar,33,0.79


In [26]:
# objectname_summary.to_csv("objectname_summary_2.csv", index=False)


In [29]:
# df_objectname_summary_2 = pd.read_csv("objectname_summary_2.csv")

In [ ]:
group_counts_case_insensitive = (
    df_objectname_summary_2["objectName_Group"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .str.casefold()
    .value_counts()
    .rename_axis("objectName_Group_normalized")
    .reset_index(name="count")
)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
    display(group_counts_case_insensitive)

,objectName_Group_normalized,count
0,jar,54
1,vase,35
2,bottle,26
3,jug,24
4,pot,18
5,flask,13
6,amphora,13
7,vessel,9
8,beaker,8
9,ewer,6


In [ ]:
# keywords = ["mold", "toy", "tumbler", "goblet", "figure", "coffeepot"]  # add/remove terms here

# keywords = ["dipper", "ink", "statuette", "tazza"]
# keywords = ["trumbler"]
# keywords = ["aidoion"]
# keywords = ["lamp"]
# keywords = ["lekanis"]
# keywords = ["tripod"]
# keywords = ["kylix", "syphos", "dinos", "lekanis", "coupe"]
# keywords = ["bottle vase", "magic straws"]
# keywords = ["waster", "roundel", "aidoion", "oon", "magic"]
# keywords = ["hydria"]
keywords = ["kalathos"]

pattern = "|".join(map(re.escape, keywords))

df.loc[
    df["objectName"].astype("string").str.strip().str.lower().str.contains(pattern, na=False),
    ["objectID", "title", "department", "objectName"],
].sort_values("objectID")

,objectID,title,department,objectName
3901,248515,Terracotta kalathos (vase with flaring lip),Greek and Roman Art,Kalathos


In [ ]:
object_ids = (
    df.loc[
        df["objectName"].astype("string").str.strip().str.lower().str.contains(pattern, na=False),
        "objectID",
    ]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

object_ids

[194246, 241273, 255580, 308521]